# Folium

## Imports

In [ ]:
import json
import pathlib

import folium
import geopandas

## Map and Basemaps

Nantes lies at latitude 47.218 and longitude -1.554. Display a Folium map centered on Nantes, with a zoom level that shows the city, on the `OpenTopoMap` topographic basemap rather than on the default OpenStreetMap one.

Add a second basemap to it, the `Esri.WorldImagery` aerial photography, and a layer switcher to go from one to the other. What does each basemap show that the other doesn't?

**Useful documentation**

- https://python-visualization.github.io/folium/latest/reference.html#folium.raster_layers.TileLayer
- https://leaflet-extras.github.io/leaflet-providers/preview/

In [ ]:
# Your code here

### Additional hints

- `folium.Map(location=..., zoom_start=..., tiles=...)`: `location` is a (latitude, longitude) pair, in that order; `zoom_start` goes from 0 (the whole world) to 18 (the street), and between 12 and 14, the city fits in the map; `tiles` takes the name of a basemap of the `xyzservices` package, such as `"OpenTopoMap"`.
- `folium.TileLayer("Esri.WorldImagery", name=...)` creates one more basemap, which `.add_to(city_map)` adds to the map; `name` is its label in the switcher.
- `folium.LayerControl()`, added last, lists the map's basemaps: only one is displayed at a time.
- Leave the CartoDB basemaps aside: they now require an API key.
- The map is displayed when it is the last expression of the cell.

```python
NANTES_COORDS = 47.218371, -1.553621

city_map = folium.Map(location=NANTES_COORDS, zoom_start=..., tiles=...)
folium.TileLayer(..., name="Aerial photography").add_to(city_map)
folium.LayerControl().add_to(...)

city_map
```

### Solution

In [ ]:
NANTES_COORDS = 47.218371, -1.553621

city_map = folium.Map(location=NANTES_COORDS, zoom_start=13, tiles="OpenTopoMap")
folium.TileLayer("Esri.WorldImagery", name="Aerial photography").add_to(city_map)
folium.LayerControl().add_to(city_map)

# The topographic basemap shows names, roads and relief; the photography shows
# buildings, vegetation and the Loire's sandbanks, without any name.
city_map

## Adding Markers

Now that we have a properly initialized map, we'll display on it usage information of Nantes' bike-sharing service: Bicloo.

Let's start by fetching the data with the cell below.

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/nantes-open-data/infos-velos.json

### Loading the Data

This data is now available in the `infos-velos.json` file. Load this file with the standard library's `json` module.

The file holds 74,000 characters, too many to read on screen: display instead the keys of the loaded dictionary, the number of stations (the `nhits` key), then the first element of the `records` list, a station. Where is the station's position? It appears twice: in which order each time, and which one suits `folium.Map(location=...)`?

**Useful documentation**

- https://docs.python.org/3/library/json.html#json.load

In [ ]:
# Your code here

#### Additional hints

- `json.load(file)` reads an open file and returns the Python object it holds: here, a dictionary, whose `.keys()` gives the keys.
- Open the file in a `with`, with `encoding="utf8"`.
- The dictionary has three keys, `nhits`, `parameters` and `records`: `records` is the list of the 129 stations.
- A station is a dictionary: `fields` holds its fields (name, stands, available bikes, position…), `geometry` its geometry in the GeoJSON format.
- Compare `fields["position"]` and `geometry["coordinates"]` with Nantes' coordinates: the latitude is around 47, the longitude around -1.5.

```python
with pathlib.Path("infos-velos.json").open(encoding="utf8") as file:
  data = ...
print(data.keys(), data[...])
first = data["records"][0]
print(first[...])
print(first["fields"][...], first["geometry"][...])
```

#### Solution

In [ ]:
with pathlib.Path("infos-velos.json").open(encoding="utf8") as file:
  data = json.load(file)
print(data.keys(), data["nhits"])
first = data["records"][0]
print(first["fields"])
# fields["position"]: [latitude, longitude], the order folium expects;
# geometry["coordinates"]: [longitude, latitude], GeoJSON's order.
print(first["fields"]["position"], first["geometry"]["coordinates"])

### Creating the Markers

You can now iterate over the data (in particular over the content under the `records` key of each data element) to display a marker at each Bicloo station position. The station's name is displayed when the mouse hovers over the marker.

**Useful documentation**

- https://python-visualization.github.io/folium/latest/reference.html#folium.map.Marker

In [ ]:
# Your code here

#### Additional hints

- Each element of `data["records"]` is a station: its position is in `station["fields"]["position"]`, a [latitude, longitude] list, the order folium expects (`station["geometry"]["coordinates"]` gives the reverse order, longitude then latitude, GeoJSON's).
- `folium.Marker(location=..., tooltip=...)` creates a marker, and its `.add_to(city_map)` method puts it on the map; `tooltip` is the text displayed on hover, here `station["fields"]["name"]`.
- Create a new map before the loop, and end the cell with `city_map` to display it: 129 markers.

```python
city_map = folium.Map(location=NANTES_COORDS, zoom_start=14)

for station in data["records"]:
  folium.Marker(location=..., tooltip=...).add_to(city_map)

city_map
```

#### Solution

In [ ]:
city_map = folium.Map(location=NANTES_COORDS, zoom_start=14)

for station in data["records"]:
  folium.Marker(
    location=station["fields"]["position"], tooltip=station["fields"]["name"]
  ).add_to(city_map)

city_map

### Customizing the Markers

Add some information to these markers:

- The color must depend on the number of bikes:
    - 0 → red
    - < 4 → orange
    - otherwise green
- Clicking a marker displays the station's address and its number of stands.

**Useful documentation**

- https://python-visualization.github.io/folium/latest/reference.html#folium.map.Icon

In [ ]:
# Your code here

#### Additional hints

- The number of available bikes is in `station["fields"]["available_bikes"]`, its address in `station["fields"]["address"]`, its number of stands in `station["fields"]["bike_stands"]`.
- `popup=` accepts HTML: `<br />` breaks the line.
- `folium.Marker` takes `popup=`, the text displayed on click, and `icon=folium.Icon(color=...)`, with the colors `"red"`, `"orange"` and `"green"`.
- You should get 83 green markers, 32 orange and 14 red.

```python
city_map = folium.Map(location=NANTES_COORDS, zoom_start=14)

for station in data["records"]:
  match station["fields"]["available_bikes"]:
    case x if x > 3:
      color = "green"
    case x if x > 0:
      color = ...
    case _:
      color = ...
  folium.Marker(
    location=station["fields"]["position"],
    popup=f"{...}<br />{...} stands",
    icon=folium.Icon(color=color),
  ).add_to(city_map)

city_map
```

#### Solution

In [ ]:
city_map = folium.Map(location=NANTES_COORDS, zoom_start=14)

for station in data["records"]:
  match station["fields"]["available_bikes"]:
    case x if x > 3:
      color = "green"
    case x if x > 0:
      color = "orange"
    case _:
      color = "red"
  folium.Marker(
    location=station["fields"]["position"],
    popup=f"{station['fields']['address']}<br />"
    f"{station['fields']['bike_stands']} stands",
    icon=folium.Icon(color=color),
  ).add_to(city_map)

city_map

### Proportional Circles (optional)

Replace the pins with circles (`folium.CircleMarker`) whose radius follows the station's number of stands (`bike_stands`), blue if the station has bikes, red otherwise, showing on hover the station's name, its available bikes and its stands.

How many stations have no bike? Which is the largest station? One station doesn't show: why?

**Useful documentation**

- https://python-visualization.github.io/folium/latest/reference.html#folium.vector_layers.CircleMarker

In [ ]:
# Your code here

#### Additional hints

- `folium.CircleMarker(location=..., radius=..., color=..., fill=True, tooltip=...)`: `radius` is in pixels, and the circle keeps its size on screen when zooming; `fill=True` fills it.
- A radius of a quarter of the number of stands goes from 0 to about 18 pixels (the largest station has 70 stands).
- The number of stands is in `station["fields"]["bike_stands"]`; count the stations without a bike in the loop.

```python
city_map = folium.Map(location=NANTES_COORDS, zoom_start=14)
no_bike = 0
for station in data["records"]:
  fields = station["fields"]
  available = fields["available_bikes"] > 0
  no_bike += not available
  folium.CircleMarker(
    location=fields["position"],
    radius=...,
    color="blue" if available else ...,
    fill=True,
    tooltip=f"{fields['name']}: {fields['available_bikes']} bikes, {...} stands",
  ).add_to(city_map)

print(no_bike, "stations without a bike")
city_map
```

#### Solution

In [ ]:
city_map = folium.Map(location=NANTES_COORDS, zoom_start=14)
no_bike = 0
for station in data["records"]:
  fields = station["fields"]
  available = fields["available_bikes"] > 0
  no_bike += not available
  folium.CircleMarker(
    location=fields["position"],
    radius=fields["bike_stands"] / 4,
    color="blue" if available else "red",
    fill=True,
    tooltip=f"{fields['name']}: {fields['available_bikes']} bikes, "
    f"{fields['bike_stands']} stands",
  ).add_to(city_map)

# 14 stations without a bike; the largest, 070-GARE DE NANTES SUD, has 70 stands;
# the test terminal 01011 has no stand: its circle has a zero radius.
print(no_bike, "stations without a bike")
city_map

## Adding a GeoJSON Layer

We'll now create a more complex map, with a GeoJSON layer.

To do so, we'll fetch two GeoJSON files: `bureaux.geojson` holds the polling places (a school, a town hall room…) of the 21 towns of the metropolitan area, 175 points; `zones.geojson` holds the areas of Nantes' 207 polling stations, each one linked to the polling place where its voters vote, which can host several stations.

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/nantes-open-data/zones.geojson
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/nantes-open-data/bureaux.geojson

### Loading the Data

Start by loading the two fetched files with the `geopandas.read_file` function.

Then join to the areas table the polling places table, without its geometry column, to get the address of each area's polling place: the `lien_lieu_vote` column of the areas table gives the place's identifier, the `idlieu_vote` column of the polling places table. The next two exercises use this join.

**Useful documentation**

- https://geopandas.org/en/stable/docs/reference/api/geopandas.read_file.html
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html

In [ ]:
# Your code here

#### Additional hints

- `geopandas.read_file("zones.geojson")` returns a `GeoDataFrame`, a `DataFrame` whose `geometry` column holds the geometries: 207 polygons for the areas, 175 points for the polling places.
- `zones_gdf.merge(other, how="left", left_on="lien_lieu_vote", right_on="idlieu_vote")` matches each area with the other table's row whose `idlieu_vote` equals its `lien_lieu_vote`, and keeps every area (`how="left"`); the result is still a `GeoDataFrame`.
- Remove the polling places' geometry with `drop(columns="geometry")`: otherwise the result has two geometry columns, and `folium.GeoJson` raises `TypeError: Object of type Point is not JSON serializable` in the next exercise.
- Both tables also have the `gid`, `commune`, `code_insee`, `nom` and `site` columns: `suffixes=("", "_place")` keeps the areas' names and suffixes the polling places' ones.
- You should get 207 areas, each with an `adresse`.

```python
zones_gdf = geopandas.read_file("zones.geojson")
places_gdf = geopandas.read_file("bureaux.geojson")
zones_places = zones_gdf.merge(
  places_gdf.drop(columns=...),
  how="left",
  left_on=...,
  right_on=...,
  suffixes=("", "_place"),
)
print(len(zones_places), zones_places["adresse"].isna().sum())
```

#### Solution

In [ ]:
zones_gdf = geopandas.read_file("zones.geojson")
places_gdf = geopandas.read_file("bureaux.geojson")
zones_places = zones_gdf.merge(
  places_gdf.drop(columns="geometry"),
  how="left",
  left_on="lien_lieu_vote",
  right_on="idlieu_vote",
  suffixes=("", "_place"),
)
print(len(zones_places), zones_places["adresse"].isna().sum())

### Displaying the Voting Areas

Use `folium.GeoJson` to display the voting areas.

For an extra challenge, display on mouse hover the name of the polling place, the polling station's number and the polling place's address (given by the join).

**Useful documentation**

- https://python-visualization.github.io/folium/latest/reference.html#folium.features.GeoJson
- https://python-visualization.github.io/folium/latest/reference.html#folium.features.GeoJsonTooltip

In [ ]:
# Your code here

#### Additional hints

- `folium.GeoJson` takes a `GeoDataFrame` directly, and `name="Voting areas"` names the layer (the next exercise uses it).
- `tooltip=folium.GeoJsonTooltip(fields=[...], aliases=[...])`: `fields` lists the columns to display, `aliases` their labels, in the same order.
- The columns: `nom` (the polling place's name, which up to six areas share), `numero_bureau` and `adresse`.

```python
city_map = folium.Map(location=NANTES_COORDS, zoom_start=13)

folium.GeoJson(
  zones_places,
  tooltip=folium.GeoJsonTooltip(fields=[...], aliases=[...]),
  name="Voting areas",
).add_to(city_map)

city_map
```

#### Solution

In [ ]:
city_map = folium.Map(location=NANTES_COORDS, zoom_start=13)

folium.GeoJson(
  zones_places,
  tooltip=folium.GeoJsonTooltip(
    fields=["nom", "numero_bureau", "adresse"],
    aliases=["Polling place", "Station no.", "Address"],
  ),
  name="Voting areas",
).add_to(city_map)

city_map

### Adding Markers for the Polling Places

Start from the code of the previous map and add a marker for each polling place of Nantes, that is each polling place an area points to. For an extra challenge, display the polling place's name and address when clicking the marker.

Finally, put the markers in a layer of their own, and add a layer switcher that shows or hides the areas and the markers separately.

**Useful documentation**

- https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iterrows.html
- https://python-visualization.github.io/folium/latest/reference.html#folium.map.FeatureGroup
- https://python-visualization.github.io/folium/latest/reference.html#folium.map.LayerControl

In [ ]:
# Your code here

#### Additional hints

- Keep the polling places whose `idlieu_vote` is in the join's `lien_lieu_vote` column, with `isin`: 49 places (the 126 others, outside Nantes, have no area).
- `iterrows()` yields (index, row) pairs; a polling place's geometry is a point: `place.geometry.y` is its latitude, `place.geometry.x` its longitude, and folium expects (latitude, longitude).
- `popup=` accepts HTML: `<br />` breaks the line.
- `folium.FeatureGroup(name="Polling places")` creates a layer: add the markers to it (`.add_to(places_layer)`) and add it to the map. `folium.LayerControl()` lists the map's named layers: markers added to the map itself aren't in it.

```python
nantes_places = places_gdf[places_gdf["idlieu_vote"].isin(...)]
places_layer = folium.FeatureGroup(name="Polling places").add_to(city_map)
for _, place in nantes_places.iterrows():
  folium.Marker(
    location=(..., ...),
    popup=f"{place.nom}<br />{...}",
  ).add_to(...)

folium.LayerControl().add_to(city_map)
```

#### Solution

In [ ]:
city_map = folium.Map(location=NANTES_COORDS, zoom_start=13)

folium.GeoJson(
  zones_places,
  tooltip=folium.GeoJsonTooltip(
    fields=["nom", "numero_bureau", "adresse"],
    aliases=["Polling place", "Station no.", "Address"],
  ),
  name="Voting areas",
).add_to(city_map)

nantes_places = places_gdf[
  places_gdf["idlieu_vote"].isin(zones_places["lien_lieu_vote"])
]
places_layer = folium.FeatureGroup(name="Polling places").add_to(city_map)
for _, place in nantes_places.iterrows():
  folium.Marker(
    location=(place.geometry.y, place.geometry.x),
    popup=f"{place.nom}<br />{place.adresse}",
    icon=folium.Icon(icon="info-sign", color="green"),
  ).add_to(places_layer)

folium.LayerControl().add_to(city_map)

city_map

## Choropleth Map of the Stations (optional)

Color Nantes' voting areas by the number of bike stations they contain.

To do so, build a `GeoDataFrame` of the stations, points made from their positions, then find each station's area through a spatial join with `zones_gdf`, count the stations of each polling station (`idbureau`), and draw the map with `folium.Choropleth`.

How many stations are in an area, and how many areas have at least one station? Why do some stations fall in no area?

**Useful documentation**

- https://geopandas.org/en/stable/docs/reference/api/geopandas.points_from_xy.html
- https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html

In [ ]:
# Your code here

### Additional hints

- `geopandas.points_from_xy(longitudes, latitudes)` builds the points: longitude first (x), the order of GeoJSON and shapely, the reverse of `fields["position"]`. `crs="EPSG:4326"` declares WGS 84 degrees, those of `zones_gdf`.
- `geopandas.sjoin(stations_gdf, zones_gdf[["idbureau", "geometry"]], predicate="within")` keeps the stations located in an area, with their area's `idbureau`.
- `groupby("idbureau").size()` counts each area's stations, and `.reset_index(name="stations")` turns it into a two-column `DataFrame`.
- `folium.Choropleth`: `geo_data=zones_gdf`, `data=` the counts, `columns=["idbureau", "stations"]` (the key, then the value), `key_on="feature.properties.idbureau"`; `nan_fill_color="lightgray"` grays the areas with no station.
- You should find 117 stations in an area, and 65 areas with at least one station.

```python
stations = data["records"]
stations_gdf = geopandas.GeoDataFrame(
  {"name": [s["fields"]["name"] for s in stations]},
  geometry=geopandas.points_from_xy(
    [s["fields"]["position"][...] for s in stations],
    [s["fields"]["position"][...] for s in stations],
  ),
  crs="EPSG:4326",
)
in_an_area = geopandas.sjoin(stations_gdf, ..., predicate="within")
counts = in_an_area.groupby(...).size().reset_index(name="stations")
print(len(in_an_area), "stations in an area,", len(counts), "areas")

city_map = folium.Map(location=NANTES_COORDS, zoom_start=13)
folium.Choropleth(
  geo_data=zones_gdf,
  data=counts,
  columns=[..., ...],
  key_on=...,
  fill_color="BuGn",
  nan_fill_color="lightgray",
  legend_name="Number of bike stations",
).add_to(city_map)
city_map
```

### Solution

In [ ]:
stations = data["records"]
stations_gdf = geopandas.GeoDataFrame(
  {"name": [s["fields"]["name"] for s in stations]},
  geometry=geopandas.points_from_xy(
    [s["fields"]["position"][1] for s in stations],
    [s["fields"]["position"][0] for s in stations],
  ),
  crs="EPSG:4326",
)
in_an_area = geopandas.sjoin(
  stations_gdf, zones_gdf[["idbureau", "geometry"]], predicate="within"
)
counts = in_an_area.groupby("idbureau").size().reset_index(name="stations")
print(len(in_an_area), "stations in an area,", len(counts), "areas")
# The 12 others are outside Nantes, in Rezé and Saint-Sébastien-sur-Loire, or
# test terminals.
print(sorted(set(stations_gdf["name"]) - set(in_an_area["name"])))

city_map = folium.Map(location=NANTES_COORDS, zoom_start=13)
folium.Choropleth(
  geo_data=zones_gdf,
  data=counts,
  columns=["idbureau", "stations"],
  key_on="feature.properties.idbureau",
  fill_color="BuGn",
  nan_fill_color="lightgray",
  legend_name="Number of bike stations",
).add_to(city_map)
city_map